In [ ]:
get_ipython().system('hostname')

In [ ]:
from photometry.models.baselines import LambertianModel, MinnaertModel
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.core.types import GeometryBatch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
import rasterio
import spiceypy as spice
import plotly.graph_objects as go
import duckdb
import os
print(os.getcwd())


In [ ]:
# Set up paths and load phase-curve data for all phases

project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent



parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "survey"/"*.parquet"

parquet_path_hamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "hamo"/"*.parquet"

parquet_path_lamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "lamo"/"*.parquet"

binned_parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "golden" / "survey_binned_dsk256_110825_range80.parquet"



dtm_path = project_root / "data" / "dtm" / "DTM_VESTA_93M.TIF"

dsk_path = project_root / "data" / "spice_kernels" / "vesta_gaskell_256_110825.bds"

In [ ]:
survey_df = pd.read_parquet(binned_parquet_path_survey_gaskell_dsk256_110825)
display(survey_df)

print(survey_df.columns.tolist())

In [ ]:
df = survey_df.copy()


df["mu0_true"] = np.cos(np.deg2rad(df["mean_incidence"]))
df["mu_true"] = np.cos(np.deg2rad(df["mean_emission"]))

# Filter out high phases if needed
df = df[df["alpha_grid"] < 80]

In [ ]:
display(df)

In [ ]:
# --- Fit A(alpha), k(alpha) per phase bin ---
fitter = LeastSquaresFitter()
results = []

for alpha_bin, g in df.groupby("alpha_grid"):

    if len(g) < 5:  # skip sparse bins, need enough i/e coverage to constrain both params
        continue

    geometry = GeometryBatch(
        incidence=np.deg2rad(g["mean_incidence"].to_numpy()),
        emission=np.deg2rad(g["mean_emission"].to_numpy()),
        phase=np.zeros(len(g)),
    )
    y = g["mean_iof"].to_numpy()
    # weights = 1/sigma convention (see fitting/least_sq.py); sigma = 1/sqrt(n_pixels)
    weights = np.sqrt(g["n_pixels"].to_numpy())

    model = MinnaertModel()
    model.parameters.update({"albedo": 0.2, "k": 0.5})

    result = fitter.fit(
        model=model,
        geometry=geometry,
        observed_reflectance=y,
        weights=weights,
    )

    if not result.metadata["success"]:
        print(f"Fit failed for alpha_bin={alpha_bin}")
        continue

    model.parameters.update(result.fitted_parameters)
    predicted = np.asarray(model.reflectance(geometry))
    resid = y - predicted
    # parameter_errors is a first-class FitResult field (1-sigma per parameter);
    # NaN means it could not be estimated for this bin (see metadata["error_estimation_warning"]).
    errors = result.parameter_errors

    results.append({
        "alpha_bin": alpha_bin,
        "A": result.fitted_parameters["albedo"], "A_err": errors["albedo"],
        "k": result.fitted_parameters["k"], "k_err": errors["k"],
        "n_bins": len(g),
        "total_pixels": g["n_pixels"].sum(),
        "weighted_rms": np.sqrt(np.average(resid**2, weights=g["n_pixels"].to_numpy())),
    })

minnaert_results = pd.DataFrame(results).sort_values("alpha_bin")
display(minnaert_results)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].errorbar(minnaert_results["alpha_bin"], minnaert_results["A"],
                 yerr=minnaert_results["A_err"], fmt="o", ms=4, color="blue")
axes[0].set_xlabel("Phase Angle Grid (deg)")
axes[0].set_ylabel("Minnaert Albedo $A(\\alpha)$")
axes[0].set_title("Empirical Phase Curve")

axes[1].errorbar(minnaert_results["alpha_bin"], minnaert_results["k"],
                 yerr=minnaert_results["k_err"], fmt="o", ms=4, color="darkorange")
axes[1].axhline(1.0, color="gray", ls="--", lw=1, label="Lambertian (k=1)")
axes[1].axhline(0.5, color="gray", ls=":", lw=1, label="Lommel-Seeliger (k=0.5)")
axes[1].set_xlabel("Phase Angle Grid (deg)")
axes[1].set_ylabel("Minnaert Exponent $k(\\alpha)$")
axes[1].set_title("Limb Darkening Behavior")
axes[1].legend()

plt.tight_layout()
plt.show()